# Drone vs Bird, baseline model training

**Where this runs:** on Google Colab, in the browser. Nothing is installed on the PC.

**Before running anything:** `Runtime` > `Change runtime type` > select **T4 GPU** > Save.

Then run the cells one by one, top to bottom.


## 1. Check that the GPU is active

Must print `True` and `Tesla T4`. If it prints `False`, the runtime type was not changed.


In [ ]:
import torch

print('GPU available:', torch.cuda.is_available())
print('Device       :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


## 2. Install the tools

About a minute. Red warnings are normal.


In [ ]:
!pip install -q ultralytics roboflow


## 3. Download the dataset

**Paste your Roboflow API key between the quotes below.** It is the only thing to edit in this notebook.

This key is personal. Never publish it on GitHub.


In [ ]:
ROBOFLOW_API_KEY = "paste_your_key_here"

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('myworkspace-0p4nk').project('drone-bird-detection-3nl79')
version = project.version(3)
dataset = version.download('yolov11')

print('Downloaded to:', dataset.location)


## 4. Check what was downloaded

Expected: classes `['Bird', 'Drone']`, and about 5418 / 1547 / 772 images.

If the counts differ, this is not the right dataset version.


In [ ]:
import yaml, glob

with open(f'{dataset.location}/data.yaml') as f:
    cfg = yaml.safe_load(f)

print('Classes         :', cfg['names'])
print('Number of classes:', cfg['nc'])
print()
for split in ['train', 'valid', 'test']:
    n = len(glob.glob(f'{dataset.location}/{split}/images/*'))
    print(f'{split:6}: {n} images')


## 5. Look at a few images

Thirty seconds, but this is what tells you whether the annotations are worth anything.
Green boxes are the annotations shipped with the dataset.


In [ ]:
import os, random, glob, cv2
import matplotlib.pyplot as plt

names = cfg['names']
images = random.sample(glob.glob(f'{dataset.location}/train/images/*'), 6)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, path in zip(axes.ravel(), images):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    label = path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    title = []
    if os.path.exists(label):
        for line in open(label):
            c, xc, yc, bw, bh = line.split()
            xc, yc, bw, bh = float(xc)*w, float(yc)*h, float(bw)*w, float(bh)*h
            p1 = (int(xc - bw/2), int(yc - bh/2))
            p2 = (int(xc + bw/2), int(yc + bh/2))
            cv2.rectangle(img, p1, p2, (0, 255, 0), 3)
            title.append(names[int(c)])
    ax.imshow(img); ax.axis('off'); ax.set_title(', '.join(title) or 'no annotation')
plt.tight_layout(); plt.show()


## 6. Train

Twenty to forty minutes on paper, closer to ninety in practice. **Do not close the tab**, Colab kills idle sessions.

`patience=15` stops training if nothing improves for 15 epochs.


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=50,
    imgsz=640,
    batch=32,
    patience=15,
    project='drone_bird',
    name='baseline_n',
)


## 7. Measure on the test split

Training reports scores on the validation split. The honest number, the one for the README,
is measured on the test split, which the model has never seen.

**Keep this output.** Published reference on this dataset: mAP50 = 0.979.


In [ ]:
metrics = model.val(split='test')

print()
print('=== RESULT ON THE TEST SPLIT ===')
print('mAP50   :', round(float(metrics.box.map50), 4))
print('mAP50-95:', round(float(metrics.box.map), 4))
print()
for i, name in enumerate(cfg['names']):
    print(f'  {name:6} mAP50 = {round(float(metrics.box.ap50[i]), 4)}')


## 8. Retrieve the work before the session dies

Colab wipes everything on close. Do this right away.

Everything goes into a single zip that lands in the PC's Downloads folder.
Unpack it into `results/` and `models/` in the project.


In [ ]:
import shutil, os
from google.colab import files

# Ultralytics writes under runs/detect/, not directly under project/.
base = '/content/runs/detect'
folder = f'{base}/drone_bird/baseline_n'

assert os.path.exists(folder), f'not found: {folder}'
print('Contents:', os.listdir(folder))

model.export(format='onnx')

# One zip with everything: weights, curves, confusion matrices,
# and the validation outputs on the test split.
shutil.make_archive('/content/results_baseline_n', 'zip', base)
files.download('/content/results_baseline_n.zip')
